In [ ]:
from imports import *
from quick_class import *
import warnings
import fnmatch
from obspy.core.inventory.inventory import read_inventory
import operator
from obspy import read
# ========================================================================================================================================================
from IPython.display import clear_output
from obspy.geodetics import locations2degrees

instrument_colors = {'B2':[227,26,28], 'KE':[178,223,138], 'AB':[166,206,227], 'BA':[202,178,214], 'AR':[255,127,0], 'TRM':[31,120,180], 'BG':[51,160,44], 'BD':[106,61,154]}
_ = [instrument_colors.update({k:list(np.array(instrument_colors[k])/255)}) for k in list(instrument_colors.keys())]
seismometer_marker = {'Guralp CMG3T 120':'o','Trillium 240':'x','Trillium Compact':'^'}

def write_pickle(file,var):
    import pickle
    with open(str(file), 'wb') as handle:
        pickle.dump(var, handle, protocol=pickle.HIGHEST_PROTOCOL)
    print('Saved to :' + str(file))
def load_pickle(file):
    import pickle
    with open(file, 'rb') as handle:
        b = pickle.load(handle)
    return b
def mirror_events(reports):
    nkeys = [n for n in list(reports[0].__dict__.keys()) if not n=='f']
    mirror = dict()
    for ni,n in enumerate(nkeys):
        skeys = list(reports[0][n].__dict__.keys())
        for si,s in enumerate(skeys):
            stanm = '.'.join([n,s]).replace('n','')
            ev0 = [k.replace('.','') for k in reports[0][n][s].events]
            ev1 = [k.replace('.','') for k in reports[1][n][s].events]
            mirror[stanm] = np.intersect1d(ev0,ev1)
    return mirror
# ================================================================================================================================
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
reportfolder = Path('/Users/charlesh/Documents/Codes/OBS_Methods/NOISE/ATACR_HPS_Comp/_DataArchive/Analysis/NetworkCoherences')
plotfolder = Path('/Users/charlesh/Documents/Codes/OBS_Methods/NOISE/ATACR_HPS_Comp/_FigureArchive/_GEN6')
bands = ['1-10','10-30','30-100']
# ================================================================================================================================
# ================================================================CODE SNIPPETS===========================================================================
# for a in cm._cmap_names_categorical:
#     display(cm.__dict__[a].resampled(4))
# for (Event,Station,Metrics,Comp) in OBS_Generator(catalog,dirs['Py_DataParentFolder']):
#     print(Event)
# for i,(Event,Station,Metrics,Comp) in zip(range(1),OBS_Generator(catalog,dirs['Py_DataParentFolder'])):
#     print(Event)
# # ======================================================================================================================================================
def smooth(d,k=10):return np.convolve(d, np.ones(k) / k, mode='same')
NoiseColors = [mcolors.to_hex(m) for m in [cm.__dict__[e].resampled(30).resampled(6).colors for e in ['devon_categorical']][0]]
# [display(c) for c in [cm.__dict__[e].resampled(70).resampled(5) for e in ['nuuk_categorical','devon_categorical','hawaii_categorical','imola_categorical','lapaz_categorical']]]
# np.array([[mcolors.to_hex(c) for c in r] for r in [cm.__dict__[e].resampled(70).resampled(5).colors for e in ['nuuk_categorical','devon_categorical','hawaii_categorical','imola_categorical','lapaz_categorical']]])
# display(catalog)
print('Stations: ' + str(len(catalog)))
# catalog = catalog.iloc[np.where(catalog.Station=='M08A')[0][0]].to_frame().T
# _ = [print('['+(str(np.round(100*100*i/len(catalog))/100)).zfill(5) + '] ' + n+ '.' +s) for i,s,n in zip(range(len(catalog)),catalog.Station,catalog.Network)]
# _ = [print(s) for s in ('[' + catalog.Network + '] | ' + catalog.Experiment).unique()]

In [ ]:
fold = Path('/Users/charlesh/Documents/Codes/OBS_Methods/NOISE/ATACR_HPS_Comp/_FigureArchive/_GEN6/StationEventPages/low_rez')
sta = catalog[catalog.StaName=='2D.OBS03'].iloc[0]

evm_sortedbymag=[sta.EventMeta[j] for j in np.argsort([e.magnitudes[0].mag for e in sta.EventMeta])]

In [45]:
np.flip(np.argsort([e.origins[0].time for e in sta.EventMeta]))

array([27,  6, 19, 26, 29, 22, 21,  5,  9,  1, 12, 25, 28,  7, 17, 14, 13,
        4,  2, 11, 24,  3, 20,  8,  0, 15, 10, 18, 16, 23])

In [47]:
catalog.to_pickle(dirs.Catalogs / 'sta_catalog_101524.pkl')

In [46]:
catalog = pd.read_pickle(dirs.Catalogs / 'sta_catalog_101524.pkl')

for stai,sta in enumerate(catalog.iloc):
    evm = [sta.EventMeta[j] for j in np.flip(np.argsort([e.origins[0].time for e in sta.EventMeta]))]
    # times = np.array([UTCDateTime.strptime(e.Name,'%Y.%j.%H.%M') for e in evm])
    # mags = np.array([e.magnitudes[0].mag for e in evm]);keeps = []
    # if np.any(times==tuhoku_time):keeps = [np.where(times==tuhoku_time)[0][0]]
    # keeps.extend(np.where((abs(times-tuhoku_time)>=7200))[0]) #  & ((mags-tuhoku_mag)<=0)
    # evm = [evm[k] for k in keeps]
    # lost = len(evm)-len(sta.EventMeta)
    # print('Lost:'+str(lost))
    catalog.at[stai,'EventMeta'] =evm
    o=1

In [44]:
sta.EventMeta[0]

2010-04-06T22:15:02.130000Z

In [ ]:
catalog.iloc[0].EventMeta

In [ ]:
fold = Path('/Users/charlesh/Documents/Codes/OBS_Methods/NOISE/ATACR_HPS_Comp/_FigureArchive/_GEN6/StationEventPages/low_rez')
# np.array([len(list((fold/c.StaName).glob('*.png')))/10 for c in catalog.iloc])
# np.array([len(list((fold/c.StaName).glob('*events*.png')))/2 for c in catalog.iloc]) #EVENTS=DONE
status = [float(len(list((fold/c.StaName).glob('*coph*.png')))/8) for c in catalog.iloc]
print('Status |'+str(np.round(100*sum(status)/len(status),2))+'%|'+'--'*37)
print(np.array(status)) #COPH=NOT DONE
print('--'*40)

In [ ]:
[mcolors.to_hex(c) for c in cm.__dict__['devon_categorical'].resampled(30).resampled(5).colors]

In [ ]:
[cm.__dict__[e].resampled(30).resampled(6) for e in ['devon_categorical']]

In [ ]:
def get_noise_metrics(stakey,noisedir):
    # _____________________________________________________________________
    # -------Get good days
    avg = load_pickle(list((noisedir.parent/'AVG_STA'/stakey).glob('*sta.pkl'))[0])
    gooddays = np.array(list((noisedir.parent/'Spectra'/stakey).glob('*.spectra.pkl')))[avg.gooddays]
    days = np.array(list((noisedir / stakey).glob('*Z.SAC')))[avg.gooddays]
    days = [d.name.replace('..','.').replace('.HZ.SAC','') for d in days]
    overlap,window=load_pickle(gooddays[0]).overlap,load_pickle(gooddays[0]).window
    goodwins = [load_pickle(g).goodwins for g in gooddays]
    for d in days:
        noise = Stream([load_sac(noisedir/stakey/''.join([d,'..',c,'.SAC']),rmresp=True)[0][0] for c in ['H1','H2','HZ','HDH']])
        n = [s for s in noise.split(window_length=window,step=window*(1-overlap))]
    # -------Build noise
    noise = Stream([load_sac(noisedir/stakey/''.join([d,'..',c,'.SAC']),rmresp=True)[0][0] for c in ['H1','H2','HZ','HDH'] for d in days])
    NoiseTr=Stream([noise.select(channel='*Z')[0].copy(),noise.select(channel='*1')[0].copy(),noise.select(channel='*2')[0].copy(),noise.select(channel='*DH')[0].copy()]).copy()
    for c in ['H1','H2','HZ','HDH']:NoiseTr.select(channel='*'+c)[0].data=np.mean(noise.select(channel='*'+c),axis=0)
    # -------Build metrics
    NoiseM=OBSM.Metrics(tr1=NoiseTr.select(channel='*1')[0],tr2=NoiseTr.select(channel='*2')[0],trZ=NoiseTr.select(channel='*Z')[0],trP=NoiseTr.select(channel='*DH')[0])
    return NoiseM


noisedir = dirs.Noise
for stai,stakey in enumerate(catalog.StaName):
    print(str(stai+1)+'/'+str(len(catalog)))
    NoiseM = get_noise_metrics(stakey,noisedir)
    write_pickle(str(noisedir.parent/'AVG_STA' / stakey / (stakey+'_NoiseMetrics.pkl')),NoiseM)
    clear_output(wait=False)

In [ ]:
noisedir.parent/'AVG_STA'

In [ ]:
noise = Stream([load_sac(noisedir/stakey/''.join([d,'..',c,'.SAC']),rmresp=True)[0][0] for c in ['H1','H2','HZ','HDH'] for d in days])
NoiseTr=Stream([noise.select(channel='*Z')[0].copy(),noise.select(channel='*1')[0].copy(),noise.select(channel='*2')[0].copy(),noise.select(channel='*DH')[0].copy()]).copy()
for c in ['H1','H2','HZ','HDH']:NoiseTr.select(channel='*'+c)[0].data=np.mean(noise.select(channel='*'+c),axis=0)
NoiseM=OBSM.Metrics(tr1=NoiseTr.select(channel='*1')[0],tr2=NoiseTr.select(channel='*2')[0],trZ=NoiseTr.select(channel='*Z')[0],trP=NoiseTr.select(channel='*DH')[0])

In [ ]:
NoiseTr=Stream([noise.select(channel='*Z')[0].copy(),noise.select(channel='*1')[0].copy(),noise.select(channel='*2')[0].copy(),noise.select(channel='*DH')[0].copy()]).copy()
for c in ['H1','H2','HZ','HDH']:NoiseTr.select(channel='*'+c)[0].data=np.mean(noise.select(channel='*'+c),axis=0)
NoiseM=OBSM.Metrics(tr1=NoiseTr.select(channel='*1')[0],tr2=NoiseTr.select(channel='*2')[0],trZ=NoiseTr.select(channel='*Z')[0],trP=NoiseTr.select(channel='*DH')[0])

In [ ]:
x,y=NoiseM.Phase('ZP')
plt.scatter(x[x<=1],y[x<=1],s=0.1)
plt.xscale('log')

In [ ]:
def get_event_list(sta,evdir,tf = 'sta.ZP-21',mirror_fold=None):
    # sta,evdir,tf = 'sta.ZP-21.HZ.SAC',mirror_fold=None
    stafold = evdir / sta.StaName
    evmeta = sta.EventMeta
    events = [f.name.replace(sta.StaName + '.','').replace('.'+tf,'') for f in list((stafold /  'CORRECTED').glob('*.'+tf))]
    if mirror_fold:
        mirror = np.unique([g[:g.find('.Z')] for g in 
        [f.name.replace(sta.StaName + '.','').replace('.sta','').replace('.day','').replace('.SA','').replace('.'+'HZ','') 
        for f in list((mirror_fold/sta.StaName/'CORRECTED').glob('*'+'HZ'+'.'+'SAC'))]])
        events = list(np.intersect1d(events,mirror))
    eind = [[np.intersect1d(events,[e.Name for e in evmeta],return_indices=True)[1]][0],[np.intersect1d(events,[e.Name for e in evmeta],return_indices=True)[2]][0]]
    evmeta = Catalog([evmeta[e] for e in eind[1]]);events = [events[e] for e in eind[0]]
    return evmeta

def get_station_events(sta,evdir,type='stream',tf = 'sta.ZP-21.HZ.SAC',mirror_fold=None):
    # sta,evdir,type='stream',tf = 'sta.ZP-21.HZ.SAC',mirror_fold=None
    stafold = evdir / sta.StaName
    # --------------------------------------------- Event list
    evmeta = get_event_list(sta=sta,evdir=evdir,tf=tf,mirror_fold=mirror_fold)

    # --------------------------------------------- Load and store data
    label=tf.replace('sta.','').replace('.SAC','')
    # For streams
    if type=='stream':
        st_hold = Stream()
        for evi,ev in enumerate(evmeta):
            raw = load_sac(stafold / (ev.Name + '.HZ.SAC'),rmresp=True)[0]
            corrected = load_sac(stafold /  'CORRECTED' / '.'.join([sta.StaName,ev.Name,tf]),rmresp=False)[0]
            raw[0].stats.location = 'Raw'
            corrected[0].stats.location = 'Corrected.'+label
            st = raw+corrected
            st.taper(0.01).filter('bandpass',freqmin=1/100,freqmax=1,zerophase=True,corners=4).taper(0.01)
            st_hold = st_hold + st
# For metrics
    elif type=='metrics':
        st_hold = Stream()
        for evi,ev in enumerate(evmeta):
            raw = Stream([load_sac(stafold / (ev.Name + '.'+c+'.SAC'),rmresp=True)[0][0] for c in ['H1','H2','HDH','HZ']])
            for i in range(len(raw)):raw[i].stats.location = 'Raw'
            corrected = Stream([raw.select(channel=c)[0].copy() for c in ['*1','*2','*H']]).copy()
            corrected+=load_sac(stafold /  'CORRECTED' / '.'.join([sta.StaName,ev.Name,tf]),rmresp=False)[0][0]
            for i in range(len(corrected)):corrected[i].stats.location = 'Corrected.'+label
            if not (len(corrected)==4) or not (len(raw)==4):print('Data missing');continue
            RtrZ=raw.select(channel='*Z')[0].copy()
            RtrZ.Metrics=OBSM.Metrics(tr1=raw.select(channel='*1')[0].copy(),tr2=raw.select(channel='*2')[0].copy(),trP=raw.select(channel='*H')[0].copy(),trZ=raw.select(channel='*Z')[0].copy())
            CtrZ=corrected.select(channel='*Z')[0].copy()
            CtrZ.Metrics = OBSM.Metrics(tr1=corrected.select(channel='*1')[0].copy(),tr2=corrected.select(channel='*2')[0].copy(),trP=corrected.select(channel='*H')[0].copy(),trZ=corrected.select(channel='*Z')[0].copy())
            st_hold+=RtrZ;st_hold+=CtrZ
    return st_hold,evmeta


plotfolder = Path('/Users/charlesh/Documents/Codes/OBS_Methods/NOISE/ATACR_HPS_Comp/_FigureArchive/_GEN6')
cat = catalog.copy()
hpsfold = Path('/Users/charlesh/Documents/Codes/OBS_Methods/NOISE/ATACR_HPS_Comp/_DataArchive/HPS_Data/Data')
atacrfold = dirs.Events
method='ATaCR';evdir = atacrfold;tf='sta.ZP-21.HZ.SAC';mirror_fold = hpsfold
# method = 'Noisecut';tf='HZ.SAC';evdir = hpsfold;#mirror_fold = atacrfold
dirs
sta = cat.iloc[0]
# del mirror_fold
st_hold,evmeta = get_station_events(sta,evdir,tf=tf,mirror_fold=atacrfold,type='metrics')

rawmetrics = st_hold.select(location='*Raw*')
correctedmetrics = st_hold.select(location='*Corrected*')
A = correctedmetrics[0].copy()
B = rawmetrics[0].copy()
AB = A.Metrics / B.Metrics

In [ ]:
# tri = 1
# tr = st[tri]
# stallaz=[tr.stats.sac.stla,tr.stats.sac.stlo,tr.stats.sac.stel]
# evllaz=[tr.stats.sac.evla,tr.stats.sac.evlo,evmeta[tri].origins[0].depth/1000]
# ar = get_arrivals(stallaz,evllaz,model = 'iasp91',phases=('ttall',)) #'PKIKP','SKS','SKKS','PKIKS','SKSSKS'
# ar

In [ ]:
st_hold

In [ ]:
# [display(cm.cmaps[e].resampled(8).resampled(4)) for e in list(['devon_categorical','hawaii_categorical'])]
# colors = np.array([cm.cmaps[e].resampled(8).resampled(4).colors for e in list(['devon_categorical','hawaii_categorical'])]).reshape((8,4))
# colors_hex = [mcolors.to_hex(c) for c in colors]

In [ ]:
# from obspy import read
# f = '/Users/charlesh/Documents/Codes/OBS_Methods/NOISE/ATACR_HPS_Comp/_DataArchive/DLOPY_Data/EVENTS/2D.OBS19/2011.065.12.31.HZ.SAC'
# st = read(f)
# print(st[0].stats.endtime - st[0].stats.starttime)
# print(st[0].stats.sampling_rate)

In [ ]:
# dldata = DL(sta)
# # Add event to object
# accept = dldata.add_event(ev, gacmin=args.mindist, gacmax=args.maxdist,depmax=args.maxdep, returned=True)
# has_data = dldata.download_data(client=wf_client, stdata=args.localdata,ndval=args.ndval, new_sr=2., t1=t1, t2=t2,returned=True, verbose=args.verb)
# dldata.calc(showplot=False)
ev_intputfolder = Path(dirs['Py_CorrectedTraces'])
stakey = '7D.M08A'


cat = catalog[catalog.StaName==stakey]
staquery_output = build_staquery(cat,staquery_output=None)
# Load catalog
events = [ev[0] if isinstance(ev,obspy.core.event.catalog.Catalog) else ev for ev in cat.iloc[0].Metadata]
evcat = Catalog();evcat.extend(events)
evtimes = [e.origins[0].time for e in evcat]

# Load data
dateformat = '%Y.%j.%H.%M'
ev = evtimes[0]
ev.strftime(dateformat)
st,inv = Stream(),[]
files = [ev.strftime(dateformat)+'.*'+c+'.SAC' for c in ['Z','1','2']]
[(st.append(tr[0]),inv.append(invi)) for tr,invi in [load_sac(ev_intputfolder / stakey / f) for f in files]];inv = inv[0]
# Plot event list
evplot = evcat.plot()
fig = inv.plot(show=False,color='r',block=True,draw=False)
plt.show()
clear_output(wait=False)
evcat.plot(fig=fig)

dldata = DL(staquery_output[staquery_output.keys()[0]])
# mindist = 5
# maxdist = 175
# maxdep = 1000
dldata.add_event(events[0], returned=True)

In [ ]:
attr_sta = AttribDict()
# staquery_output.to_dict()
attr_sta.update

In [ ]:
staquery_output[staquery_output.keys()[0]]

In [ ]:
def plot_spec_coh_adm_ph(Metrics):
    pairs = ['ZP']
    meters = ['psd','Coherence','Admittance','Phase']
    fig, axes = plt.subplots(nrows=4, ncols=1,figsize=(8,10),layout='constrained',squeeze=False,sharey='row',sharex='all')
    axes = axes.reshape(-1)
    stam = Metrics['ATaCR'].traces[0].stats.network + '.' + Metrics['ATaCR'].traces[0].stats.station
    label = Metrics['ATaCR'].traces.select(channel='*Z')[0].stats.location
    Pre = Metrics['Raw']
    Post = Metrics['ATaCR']
    Noise = Metrics['Noise']
    fn = fnotch(abs(Post.traces[0].stats.sac.stel*1000))
    tstamp = Pre.traces[0].stats.starttime.strftime('%Y.%j.%H.%M')
    for pi,(ax,m) in enumerate(zip(axes,meters)):
        if m=='psd':
            p = 'Z'
        else:
            p = pairs[0]
        evf,prey = Pre.__getattribute__(m)(p)
        evf,posty = Post.__getattribute__(m)(p)
        if m=='psd':
            noisef,noisey = Noise.f,Noise.StaNoise.power.__dict__['cZZ']
        else:
            noisef,noisey = Noise.__getattribute__(m)(p)
        noisey = noisey[noisef>0]
        noisef = noisef[noisef>0]
        if m=='psd':
            noisey = 10*np.log10(noisey)
            prey = 10*np.log10(prey)
            posty = 10*np.log10(posty)
        ax.scatter(noisef,noisey,s=0.5,c='gray',label='Noise')
        if pi==0:
            lbl = stam + ' | ' + label + ' ' + tstamp + ' | '
        else:
            lbl = ''
        ax.set_xlabel('Frequency')
        ax.set_ylabel(m.replace('psd','Power Density'))
        ax.set_title(lbl + p + '-' + m.replace('psd','PSD'),fontweight='bold')
        ax.scatter(evf,prey,c='k',label='PRE',marker='o',s=1)
        ax.scatter(evf,posty,c='m',label='POST',marker='o',s=0.5)
        ax.axvline(fn,linewidth=0.2,color='k')
        if pi==0:
            ax.text(fn*1.05,0.99*min(ax.get_ylim()),'Fn:' + str(round(1/fn*100)/100) + 's',alpha=0.4)
            ax.set_xscale('log')
            ax.set_xlim(evf[1],evf[-1])
            ax.legend(markerscale=10,ncols=len(meters))
    plt.tight_layout()
    return fig

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ++++++++++++++++++++++++++ CONSTRUCTOR AREA ++++++++++++++++++++++++++++++
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
methods = ['PostATACR']
atacrdatafolder = archive / 'ATaCR_Data' / 'ATaCR_Python'
for correction_method in methods:
    coh_comp = correction_method.replace('PostHPS','HPS').replace('PostATACR','ATaCR')
    if correction_method=='PostHPS':
      return_hps = True
    else:
      return_hps = False
    OutFolder = Path(plotfolder)
    SubFolders = Path('EventRecords') / correction_method / 'coherence'
    OutFolder = OutFolder / SubFolders
    OutFolder.mkdir(parents=True,exist_ok=True)
    for station in catalog.iloc:
      stations = [station.Station]
      networks = [station.Network]
      events = station.Events
      for i,(net,sta) in enumerate(zip(networks,stations)):
        Metrics = []
        for evi,event in enumerate(events):
          depth = round(station.Metadata[evi].origins[0].depth/1000)
          mag = station.Metadata[evi].magnitudes[0].mag
          File = '.'.join([net,sta]) + '.m' + str(mag) + '.z' + str(depth) + 'km' + '.' + event + '.' + correction_method.replace('Post','') + '_SPECCOHPHADM.png'
          title = File.replace('_',' | ').replace('z','z: ').replace('m','mag: m')
          print('[' + str(evi) + '/' + str(len(events)) + '] ' + File)
          post_record = Stream()
          pre_record = Stream()
          M,Comp = get_metrics_comp(net,sta,atacrdatafolder,event,return_hps=return_hps,events_folder='EVENTS')
          # M['Noise'] = get_Noise(atacrdatafolder,net,sta,'sta')['Noise']
          Metrics.append(M.copy())
          fig = plot_spec_coh_adm_ph(M)
          save_tight(str(plotfolder / 'MeetingFigs' / 'SPECCOHPHADM' / File),fig,dpi=600)


In [ ]:
bplot['boxes'][0]._edgecolor